In [6]:
# --- MODEL 3:SVM (Support Vector Machine, 1K Sample) ---

import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings("ignore")

# 1. Load and preprocess dataset

df = pd.read_csv("processed_cardio_dataset.csv", sep=None, engine='python')

# Drop id if present and convert booleans
if 'id' in df.columns:
    df = df.drop('id', axis=1)
df = df.replace({True: 1, False: 0})

label_col = 'cardio'
X = df.drop(label_col, axis=1)
y = df[label_col]


sample_size = 40000
if len(X) > sample_size:
    df_sample = df.sample(sample_size, random_state=42)
    X = df_sample.drop(label_col, axis=1)
    y = df_sample[label_col]

# Split data (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


# 2. Baseline  SVM (Linear Kernel)

svm_base = SVC(kernel='linear', C=1, probability=False, random_state=42)
svm_base.fit(X_train_s, y_train)
yp_base = svm_base.predict(X_test_s)

print("=== BASELINE SVM (Linear Kernel) ===")
print("Accuracy:", round(accuracy_score(y_test, yp_base), 4))
print("F1 Score:", round(f1_score(y_test, yp_base), 4))
print("Classification Report:\n", classification_report(y_test, yp_base))
print("-" * 50)


# 3. Quick Parameter Tuning (tiny grid)

param_grid = {'C': [0.1, 1], 'kernel': ['linear', 'rbf']}
grid = GridSearchCV(
    SVC(probability=False, random_state=42),
    param_grid=param_grid,
    cv=2,
    scoring='f1',
    n_jobs=-1
)

print("Running FAST GridSearchCV (1K data, tiny grid)...")
grid.fit(X_train_s, y_train)
best_svm = grid.best_estimator_


# 4. Evaluate Tuned Model

yp_best = best_svm.predict(X_test_s)
print("\n=== TUNED FAST SVM ===")
print("Best Params:", grid.best_params_)
print("Accuracy:", round(accuracy_score(y_test, yp_best), 4))
print("F1 Score:", round(f1_score(y_test, yp_best), 4))
print("Classification Report:\n", classification_report(y_test, yp_best))
print("-" * 50)


# 5. Conclusion

print("OBSERVATIONS")
print("Linear SVM executes within seconds; RBF may slightly improve F1.")
print("Used 2-fold CV and minimal grid for speed.")
print("Suitable for quick testing or demonstration.")
print("SVM model finished successfully.")


=== BASELINE SVM (Linear Kernel) ===
Accuracy: 0.7256
F1 Score: 0.6933
Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.82      0.75      4047
           1       0.77      0.63      0.69      3953

    accuracy                           0.73      8000
   macro avg       0.73      0.72      0.72      8000
weighted avg       0.73      0.73      0.72      8000

--------------------------------------------------
Running FAST GridSearchCV (1K data, tiny grid)...

=== TUNED FAST SVM ===
Best Params: {'C': 0.1, 'kernel': 'rbf'}
Accuracy: 0.7305
F1 Score: 0.7126
Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.78      0.75      4047
           1       0.75      0.68      0.71      3953

    accuracy                           0.73      8000
   macro avg       0.73      0.73      0.73      8000
weighted avg       0.73      0.73      0.73      8000

---------------------------